# Imports

In [1]:
import pandas as pd
import sys
import os
import plotly.express as px
from tqdm import tqdm
import numpy as np
sys.path.append('../..')
from scipy.sparse import csgraph, csr_matrix
from definitions import DATA_DIR, ROOT_DIR, KGS_DIR
from functools import reduce
from plotly.graph_objs import Figure
import plotly.io as pio

In [2]:
FIGURES_DIR = os.path.join(ROOT_DIR, 'assets', 'graph_statistics')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [3]:
pio.templates.default = "seaborn"

In [4]:
def save_fig(fig : Figure, filename : str) -> None:
    fig.write_image(os.path.join(FIGURES_DIR, filename))

In [5]:
def show(fig : Figure) -> None:
    fig.update_layout(font_size = 12)
    fig.show()

# Utils

In [6]:
def load_graph(
    corpus : str,
    predicate_min_occ : int = 30,
    entity_min_occ : int = 2
) -> pd.DataFrame: 
    
    path = os.path.join(KGS_DIR, corpus, 'entities_mapping', 'data.jsonl')
    kg = pd.read_json(path, lines=True)

    predicates = kg['predicate'].value_counts()
    entities = pd.Series(data = kg['head'].to_list() + kg['tail'].to_list()).value_counts()

    predicates = predicates.index[predicates >= predicate_min_occ]
    entities = entities.index[entities >= entity_min_occ]

    mask = (
        kg['head'].isin(entities) &
        kg['tail'].isin(entities) &
        kg['predicate'].isin(predicates)
    )

    return kg[mask]

In [7]:
def melt_df(kg : pd.DataFrame) -> pd.DataFrame:

    melted = []

    for i,(_,row) in tqdm(enumerate(kg.iterrows()), total=len(kg)):
                        
        for key in ["head","tail"]:

            melted.append({
                "triple_id" : 2 * i + int(key == "head"),
                "entity" : row[key],
                "predicate" : row["predicate"],
                "head" : key == "head",
                "tail" : key == "tail",
                "document" : row['sentence']['document']['id'] if row['sentence'] is not np.nan else None
            })

    melted = pd.DataFrame(melted)

    return melted

In [8]:
def get_components(df : pd.DataFrame) -> pd.Series:

    entities = list(set(df['head'].to_list() + df['tail'].to_list()))
    entities_map = { e : i for i,e in enumerate(entities) }
    head = df['head'].map(entities_map).to_numpy()
    tail = df['tail'].map(entities_map).to_numpy()

    N = len(entities)
    indices = np.stack([head, tail]).transpose()
    data = np.ones(len(df))

    adj = csr_matrix(
        (data, (indices[:, 0], indices[:, 1])),
        shape=(N, N),
        dtype=int
    )

    _, labels = csgraph.connected_components(adj, directed=False)

    labels = pd.Series(
        index=entities,
        data=labels
    )

    return labels

In [9]:
def describe_graph(kg : pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.Series]:

    ### Metl-Graph
    melted = melt_df(kg)
    melted = melted[~melted['document'].isna()]

    ### Node-Level
    print("Node level")
    node_level : list[pd.Series] = [
        melted['document'].groupby(melted['entity']).agg(lambda x : len(set(x))).rename('number-of-documents'),
    ]

    melted = melted.drop_duplicates(subset=["triple_id"])

    node_level.extend([
        melted['head'].groupby(melted['entity']).sum().rename('out-degree'),
        melted['tail'].groupby(melted['entity']).sum().rename('in-degree'),
        melted['entity'].value_counts().rename('degree'),
    ])

    node_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        node_level,
        pd.DataFrame(index = node_level[0].index)
    )

    ### Relation-Level
    print("Predicate Level")
    relation_level : list[pd.Series] = [
        kg['predicate'].value_counts().rename('number-of-triples')
    ]
    
    relation_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        relation_level,
        pd.DataFrame(index = relation_level[0].index)
    )

    ### Document Level
    print("Document Level")
    document_level : list[pd.Series] = [
        melted['document'].value_counts().rename('number-of-triples'),
        melted['entity'].groupby(by = melted['document']).agg(lambda x : len(set(x))).rename('number-of-entities')
    ]

    document_level : list[pd.Series] = reduce(
        lambda a,b : a.join(b.to_frame()),
        document_level,
        pd.DataFrame(index = document_level[0].index)
    )
    

    ### Components
    print("Connected Componenets")
    components = get_components(kg)
    component_level = [
        components.value_counts().rename('number-of-entities'),
        kg['head'].map(components).value_counts().rename('number-of-triples')
    ]

    component_level = reduce(
        lambda a,b : a.join(b.to_frame()),
        component_level,
        pd.DataFrame(index = component_level[0].index)
    )

    ### Graph-Level
    print("Graph Level")
    graph_level = dict()

    graph_level['Number of Triples'] = len(kg)
    graph_level['Number of Entities'] = melted['entity'].nunique()
    graph_level['Number of Predicates'] = melted['predicate'].nunique()

    graph_level['Avg. In-Degree'] = node_level['in-degree'].mean()
    graph_level['Avg. Out-Degree'] = node_level['out-degree'].mean()
    graph_level['Avg. Degree'] = node_level['degree'].mean()
    graph_level['Avg. Documents / Entity'] = node_level['number-of-documents'].mean()

    graph_level['Avg. Triples / Predicate'] = relation_level['number-of-triples'].mean()
    graph_level['Avg. Triples / Document'] = document_level['number-of-triples'].mean()
    graph_level['Avg. Entity / Document'] = document_level['number-of-entities'].mean()

    graph_level['Largest Connected Component (Number of Entities)'] = component_level['number-of-entities'].max()
    graph_level['Largest Connected Component (Number of Triples)'] = component_level['number-of-triples'].max()
    graph_level['Number Of Connected Components'] = len(component_level)

    graph_level = pd.Series(graph_level)

    return node_level, relation_level, document_level, component_level, graph_level

# Load the data

In [86]:
kg = load_graph('industry_of_future', entity_min_occ = 0, predicate_min_occ = 0).dropna(axis=0)
kg = kg.drop_duplicates(subset='id')
kg.head()

,id,sentence,head,predicate,tail,predicate_group
0,542065800,"{'id': '5420658_0', 'sentence': 'The construct...",construction sector,contribute to,global greenhouse gas emission,contributes
4,542065810,"{'id': '5420658_1', 'sentence': 'This study ev...",machine learning model,predict,global warming potential,predict
5,542065830,"{'id': '5420658_3', 'sentence': 'Data preproce...",data preprocessing,involve,feature selection,involves
6,542065831,"{'id': '5420658_3', 'sentence': 'Data preproce...",data preprocessing,involve,management of missing value,involves
7,542065832,"{'id': '5420658_3', 'sentence': 'Data preproce...",data preprocessing,involve,outlier removal,involves


In [88]:
mask = kg['sentence'].apply(lambda x : max([item['score'] for item in x['document']['pseudo_labels']['llm:deepseek-chat']]))
mask = mask / mask.max()
mask = mask.gt(0.5)
# mask = kg['sentence'].apply(lambda x : len(x['document']['labels']) > 0)

In [89]:
sub_kg = kg[mask]

In [90]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(kg)

100%|██████████| 106545/106545 [00:07<00:00, 13594.18it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [91]:
graph_level

Number of Triples                                   106545.000000
Number of Entities                                  101723.000000
Number of Predicates                                 13073.000000
Avg. In-Degree                                           1.047403
Avg. Out-Degree                                          1.047403
Avg. Degree                                              2.094806
Avg. Documents / Entity                                  1.373023
Avg. Triples / Predicate                                 8.150004
Avg. Triples / Document                                 26.002441
Avg. Entity / Document                                  17.043075
Largest Connected Component (Number of Entities)     58198.000000
Largest Connected Component (Number of Triples)      75389.000000
Number Of Connected Components                       14473.000000
dtype: float64

In [92]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg)

100%|██████████| 94831/94831 [00:04<00:00, 23484.62it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [93]:
graph_level

Number of Triples                                   94831.000000
Number of Entities                                  91051.000000
Number of Predicates                                11839.000000
Avg. In-Degree                                          1.041515
Avg. Out-Degree                                         1.041515
Avg. Degree                                             2.083030
Avg. Documents / Entity                                 1.364708
Avg. Triples / Predicate                                8.010052
Avg. Triples / Document                                26.341944
Avg. Entity / Document                                 17.258056
Largest Connected Component (Number of Entities)    51907.000000
Largest Connected Component (Number of Triples)     66893.000000
Number Of Connected Components                      13019.000000
dtype: float64

In [94]:
components = get_components(sub_kg)

In [95]:
components = get_components(sub_kg)
largest_cc = components.value_counts().idxmax().item()
largest_cc_entities = components.index[components == largest_cc]
lcc_mask = sub_kg['head'].isin(largest_cc_entities) & sub_kg['tail'].isin(largest_cc_entities)
lcc_node_level, lcc_relation_level, lcc_document_level, lcc_component_level, lcc_graph_level = describe_graph(sub_kg[lcc_mask])

100%|██████████| 66893/66893 [00:02<00:00, 24234.68it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [96]:
lcc_graph_level

Number of Triples                                   66893.000000
Number of Entities                                  51907.000000
Number of Predicates                                 8650.000000
Avg. In-Degree                                          1.288709
Avg. Out-Degree                                         1.288709
Avg. Degree                                             2.577417
Avg. Documents / Entity                                 1.590286
Avg. Triples / Predicate                                7.733295
Avg. Triples / Document                                19.448466
Avg. Entity / Document                                 11.999855
Largest Connected Component (Number of Entities)    51907.000000
Largest Connected Component (Number of Triples)     66893.000000
Number Of Connected Components                          1.000000
dtype: float64

In [102]:
robotics_mask = sub_kg['sentence'].apply(lambda x :  
    [item['score'] for item in x['document']['pseudo_labels']['llm:deepseek-chat'] if item['label'] == 'Robotics'][0]
)
robotics_mask = robotics_mask / robotics_mask.max()
robotics_mask = robotics_mask.gt(0.5)

In [103]:
node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg[robotics_mask & lcc_mask])

/tmp/ipykernel_10929/3339035813.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  node_level, relation_level, document_level, component_level, graph_level = describe_graph(sub_kg[robotics_mask & lcc_mask])
100%|██████████| 7232/7232 [00:00<00:00, 23666.57it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [104]:
graph_level

Number of Triples                                   7232.000000
Number of Entities                                  6237.000000
Number of Predicates                                1643.000000
Avg. In-Degree                                         1.159532
Avg. Out-Degree                                        1.159532
Avg. Degree                                            2.319064
Avg. Documents / Entity                                1.393939
Avg. Triples / Predicate                               4.401704
Avg. Triples / Document                               19.388740
Avg. Entity / Document                                11.654155
Largest Connected Component (Number of Entities)    4189.000000
Largest Connected Component (Number of Triples)     5584.000000
Number Of Connected Components                       494.000000
dtype: float64

In [105]:
sub_kg = sub_kg[robotics_mask & lcc_mask]
components = get_components(sub_kg)
largest_cc = components.value_counts().idxmax().item()
largest_cc_entities = components.index[components == largest_cc]
robotics_lcc_mask = sub_kg['head'].isin(largest_cc_entities) & sub_kg['tail'].isin(largest_cc_entities)
lcc_node_level, lcc_relation_level, lcc_document_level, lcc_component_level, lcc_graph_level = describe_graph(sub_kg[robotics_lcc_mask])

/tmp/ipykernel_10929/3248472699.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sub_kg = sub_kg[robotics_mask & lcc_mask]
100%|██████████| 5584/5584 [00:00<00:00, 22253.99it/s]


Node level
Predicate Level
Document Level
Connected Componenets
Graph Level


In [106]:
lcc_graph_level

Number of Triples                                   5584.000000
Number of Entities                                  4189.000000
Number of Predicates                                1298.000000
Avg. In-Degree                                         1.333015
Avg. Out-Degree                                        1.333015
Avg. Degree                                            2.666030
Avg. Documents / Entity                                1.532585
Avg. Triples / Predicate                               4.302003
Avg. Triples / Document                               16.718563
Avg. Entity / Document                                 9.610778
Largest Connected Component (Number of Entities)    4189.000000
Largest Connected Component (Number of Triples)     5584.000000
Number Of Connected Components                         1.000000
dtype: float64

# Visualization

## Node-Level 

- In-Degrees

In [20]:
fig = px.histogram(
    data_frame = node_level,
    x = "in-degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "In-Degree", yaxis_title = "Number of Entities", font_size=17)

save_fig(fig, 'in_degree_distribution.pdf')
show(fig)

- Out Degree

In [21]:
fig = px.histogram(
    data_frame = node_level,
    x = "out-degree",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = "Out-Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'out_degree_distribution.pdf')
show(fig)

- Degree

In [22]:
fig = px.histogram(
    data_frame = node_level,
    x = "degree",
    width = 800,
    log_y=True
)

fig.update_layout(xaxis_title = "Degree", yaxis_title = "Number of Entities")
save_fig(fig, 'node_degree_distribution.pdf')
show(fig)

- Most frequent entities

In [23]:
fig = px.bar(
    data_frame = node_level['degree'].sort_values(ascending = False).reset_index()[:20][::-1],
    y = "entity",
    x = "degree",
    width = 800,
)

fig.update_layout(xaxis_title = "Number of Triplets", yaxis_title = "Entity")
save_fig(fig, 'top_10_most_frequent_entities.pdf')
show(fig)

- Predicates

In [24]:
fig = px.histogram(
    data_frame = relation_level.reset_index(),
    x = "number-of-triples",
    width = 800,
    log_y=True
)
fig.update_layout(xaxis_title = 'Relation Frequency', yaxis_title = "Number Of Relations")
save_fig(fig, 'relations_frequency_distribution.pdf')
show(fig)

In [25]:
fig = px.bar(
    data_frame = relation_level.sort_values(by = "number-of-triples", ascending = False).reset_index()[:20][::-1],
    y = "predicate",
    x = "number-of-triples",
    width = 800,
    log_x=True
)
fig.update_layout(xaxis_title = 'Number of Triplets', yaxis_title = "Predicate")
save_fig(fig, 'top_10_most_frequent_predicates.pdf')
show(fig)

In [33]:
kg[kg['sentence'].isna()]['predicate'].value_counts()

predicate
use                          4007
skosbroader be hyponym of    2774
include                       911
analyze                       885
base on                       494
produce                       484
solve                         287
improve                       258
acquire                       241
match                         141
provide                       140
support                       134
execute                       123
limit                          77
propose                        66
identifie                      37
transfer                       31
learn                          30
affect                         20
link                           17
classifie                      16
predict                        16
contribute to                  15
manage                         15
extract                        12
interact with                  10
outperform                      6
guide                           6
process                         5
caus